# IRS PDF Form Build Keys

------------------
## form1065_filler.py
Python class that fills an IRS Form 1065 PDF (U.S. Return of Partnership Income)
from a plain Python dict.


## Workflow: FormKey Mapping

#### A. Gen Key Maps (irs.irsForms)

1. genTestKeyDict
    - irsForm -> fldDict [Field Dict]
    - iraForm -> testDict
    - OUT:
        - testDict
        - FILE: pdfFill(testDict) -> irsForms/<fm>-IRS.pdf.  [visual of forms with F#]
5. genIRSMap
    - fldDict -> keyDict -> irsForms/<fm>-IRS.json [map <fldName> -> F# ]

#### B. Gen IRS Report (irs.pdfFill)

1. gen_glTaxDict
    - glDict -> glKeys. [ Core of IRS Tax mapping]
    - glDict X glKeys -> glTaxDict -> irsForms/<fm>_glKey.json [<fldName -> glDict.alues]
    - glTaxDict X keyDict -> fmDict  [form values: fmKey -> glDict.values]
1. FILE: pdfFill(fmDict) ->  YE_Tax_Report/<fm>-IRS.pdf

------
## Build Key

- input IRS_Forms/formFN.pdf
- get all fields into Dict
- save 

In [2]:
# LLC GL data
import os
from pathlib import Path
import json
from ledger.LLC import LLC
from irs.pdfFill import pdfFill
from IPython.display import display, Markdown

# Link to LLC ledgers
top = Path.cwd().parents[2]
llcName = [f for f in os.listdir(top) if 'llcProfile' in f][0].replace('.json','').replace('llcProfile_','')
llc = LLC(llcName,debug=False, top=top)
# Save entity information
eDict = llc.entity
acctDIR = llc.acctDir() #os.path.join(llc.TOP, llc.dirAccounting, str(llc.yr))
yeDIR = llc.acctDir(dirName='ye')


# ----------   initialize IRS forms
from irs.irsForms import irsF1065, irsSchK1
           
fmSchK1 = irsSchK1(llc)
fm1065 = irsF1065(llc)

fm = fmSchK1

## irsFormWorkFlow - form-IRS.pdf -> Form-keys.pdf

#### Input:  Form-FieldNames.json

- option: genFldNames=True| will generate (**replace**) Form-FieldNames.json


In [3]:
#irsFormWorkFlow - form-IRS.pdf -> Form-keys.pdf, using Form-FieldNames.json
def irsFormWorkFlow(fm, **kwargs):
    debug = kwargs.get('debug', False)

    
    
    display(Markdown(f"##### \n## Notebook using:\n### LLC: {llc.entity['entity_name']}\n\n### IRS Form: {fm.oID}"))
    inFN = fm.inFN()
    outFN = fm.outFN()
    keyFN = fm.keyFN()
    fldNmFN = fm.FN('FieldNames.json')
    print("<PDF IN     :", Path(inFN).name)
    print("<JSON FldNm :", Path(fldNmFN).name)
    print(">JSON Key   :", Path(outFN).name)
    print(">PDF OUT    :", Path(keyFN).name)
    
    
    ## Per Tax Year - Do once
    
    # Per Year, Save Form json field names, modify if any changes, json save per year
    # Do this once per year to reconcile new forms to IRS definitions.
    genFldNames = kwargs.get('genFldNames', False)
    if genFldNames: 
        if fm.oID == 'fmSchK1':
            from irs.irsFormFieldNames import irsSchKFields
            fm.fldNmSave(irsSchKFields().fldNmDict)
        elif fm.oID == 'irsF1065':
            from irs.irsFormFieldNames import irsF1065Fields
            fm1065.fldNmSave(irsF1065Fields().fldNmDict)
        print(f"Create key.json: {fm.oID}")
    
    ## Get Dict of all fields in form :: fm.genTestKeyDict()

    yeDIR = fm.llc.acctDir(to = 'ye')
    irsFormDir = os.path.join(yeDIR, 'Forms_IRS')
    
    #---- 1. Get form fields
    pf = pdfFill(inFN, outFN)
    fDict = pf.get()
    print(f"\nStep 1: Loaded form Field Name")
    if debug: 
        print(f"Sample fDict\n{list([(k,d) for k,d in testDict.items()])[0:3]}")
    
    # --- 2. Load Form map (json) : field index matches order offDict.keys()
    fldFN = fldNmFN
    fldNmDict = fm.fldNmLoad()
    # Load from py
    #fldNmDict = irsF1065Fields().fldNmDict
    print(f"\nStep 2: Load form Field Keys: key.fld:{len(fldNmDict)}/fm.fld{len(fDict)}, FILE: {fldFN}")
    if debug: 
        print(f"View fldNmDict\n{list([(k,d) for k,d in fldNmDict.items()])[0:3]}")
    
    # ---- 3. Create testDict to map field ID (F#) into every field
    #         OLD testDict = {k:self.pf._testKey(i,k,fDict[k]) for i,k in enumerate(fDict)}
    fldNmDict = fm.fldNmLoad()
    testDict =  {}
    for (i,k1),(j,k2) in zip(enumerate(fDict), enumerate(fldNmDict)):
        d = fDict[k1]
        d['field_id'] = k2
        testDict[k1] = d
    print(f"\nStep 3: Create Map: : testDict:{len(testDict)} -> keyNm:{len(fldNmDict)} x fm.fld:{len(fDict)}")
    if debug: 
        print(f"View testDict\n{list([(k,d) for k,d in testDict.items()])[0:3]}")
    
    
    # ---- 4. Output: PDF key.pdf  
    ts = pdfFill(inFN, outFN)
    oDict = ts.fillPDF(to = testDict, )
    print(f"\nStep 4: Create outPDF") 
    
    print(f"\n{fm.__class__.__name__} SUCC: testDict Generated {len(testDict)} Fields; \n-- FILE: {Path(keyFN)}")

irsFormWorkFlow(fm1065)  # genFldNames=True|False
print("\n\n", '*'*60, "\n\n")
irsFormWorkFlow(fmSchK1)  # genFldNames=True|False
     
    
      
    

##### 
## Notebook using:
### LLC: W&B Group, LLC

### IRS Form: irsF1065

<PDF IN     : Form_1065-IRS.pdf
<JSON FldNm : Form_1065-FieldNames.json
>JSON Key   : Form_1065-keys.pdf
>PDF OUT    : Form_1065-keys.json

Step 1: Loaded form Field Name
Loading Form Field Name (json): Form_1065-FieldNames.json
DIR: /Users/frankrojas/GDrive/Family/Assets-Hobby/RealEstateInvestments/LLC-WB-Group/pages/AccountingData/2025/YE_Tax_Records/Forms_IRS/Form_1065-FieldNames.json

Step 2: Load form Field Keys: key.fld:440/fm.fld440, FILE: /Users/frankrojas/GDrive/Family/Assets-Hobby/RealEstateInvestments/LLC-WB-Group/pages/AccountingData/2025/YE_Tax_Records/Forms_IRS/Form_1065-FieldNames.json
Loading Form Field Name (json): Form_1065-FieldNames.json
DIR: /Users/frankrojas/GDrive/Family/Assets-Hobby/RealEstateInvestments/LLC-WB-Group/pages/AccountingData/2025/YE_Tax_Records/Forms_IRS/Form_1065-FieldNames.json

Step 3: Create Map: : testDict:440 -> keyNm:440 x fm.fld:440

Step 4: Create outPDF

irsF1065 SUCC: testDict Generated 440 Fields; 
-- FILE: /Users/frankrojas/GDrive/Famil

##### 
## Notebook using:
### LLC: W&B Group, LLC

### IRS Form: irsSchK1

<PDF IN     : Schedule_K_1-IRS.pdf
<JSON FldNm : Schedule_K_1-FieldNames.json
>JSON Key   : Schedule_K_1-keys.pdf
>PDF OUT    : Schedule_K_1-keys.json

Step 1: Loaded form Field Name
Loading Form Field Name (json): Schedule_K_1-FieldNames.json
DIR: /Users/frankrojas/GDrive/Family/Assets-Hobby/RealEstateInvestments/LLC-WB-Group/pages/AccountingData/2025/YE_Tax_Records/Forms_IRS/Schedule_K_1-FieldNames.json

Step 2: Load form Field Keys: key.fld:111/fm.fld111, FILE: /Users/frankrojas/GDrive/Family/Assets-Hobby/RealEstateInvestments/LLC-WB-Group/pages/AccountingData/2025/YE_Tax_Records/Forms_IRS/Schedule_K_1-FieldNames.json
Loading Form Field Name (json): Schedule_K_1-FieldNames.json
DIR: /Users/frankrojas/GDrive/Family/Assets-Hobby/RealEstateInvestments/LLC-WB-Group/pages/AccountingData/2025/YE_Tax_Records/Forms_IRS/Schedule_K_1-FieldNames.json

Step 3: Create Map: : testDict:111 -> keyNm:111 x fm.fld:111

Step 4: Create outPDF

irsSchK1 SUCC: testDict Generated 111 Fields; 
-- FILE: /Us

In [ ]:
print(len(glDict), len(testDict))
d = {}
for (i,k1),(j,k2)  in  zip(enumerate(glDict), enumerate(testDict)):
    g = glDict[k1]
    fldDict = testDict[k2]
    f = fldDict['field_id']
    if f != g : 
        print("ERROR: glDict and fmDict have a mismatch")    
        print("--ERR>", i,j, glDict[k1], testDict[k2])
    v = '' if fldDict['type'] == 'text' else 'chk'
    d[k1] = v
list([(k,d) for k,d in d.items()])[0:3]
fn = fm.FN('glKeys.json')
print(fn)
with open(fn, 'w') as fio:
    json.dump(d, fio, indent=4)